# Legacy Least Squares Registry Demo

This notebook requires a running Model Zoo Registry API. Set `MODEL_ZOO_API_BASE` to the service URL (defaults to `http://localhost:8000`).

The steps below register the raw legacy configuration, convert it into binding-friendly variants, author a fresh canonical JSON configuration, and execute runs for each environment via the live `/registry/configs` and `/runs` endpoints.

In [110]:
import json
import os
from copy import deepcopy
from typing import Any, Dict, Optional

import requests

API_BASE = os.getenv("MODEL_ZOO_API_BASE", "http://localhost:8000").rstrip("/")
print(f"Using live Model Zoo API at {API_BASE}")

class LiveClient:
    def __init__(self, base_url: str) -> None:
        self.base_url = base_url.rstrip("/")

    def post(self, path: str, json: Dict[str, Any]) -> "ResponseWrapper":
        response = requests.post(f"{self.base_url}{path}", json=json, timeout=60)
        response.raise_for_status()
        return ResponseWrapper(response)

    def get(self, path: str, params: Optional[Dict[str, Any]] = None) -> "ResponseWrapper":
        response = requests.get(f"{self.base_url}{path}", params=params, timeout=60)
        response.raise_for_status()
        return ResponseWrapper(response)

class ResponseWrapper:
    def __init__(self, response: requests.Response) -> None:
        self._response = response

    def json(self) -> Any:
        return self._response.json()

client = LiveClient(API_BASE)


Using live Model Zoo API at http://localhost:8000


## Register the Raw Legacy Payload

In [111]:
variable_config_csv = """COLUMN,CATEGORIZATION,VARIABLE LABEL,DUE TO GROUP,SUMMARY GROUP,TYPE,UPPER CHANNEL LABEL,CHANNEL LABEL,SUB CHANNEL LABEL,INCLUDE,DECOMP TYPE,FIXED INTERACTIONS,RANDOM INTERACTIONS,SIGN EXPECTATION,LOWER BOUND,UPPER BOUND,PRIOR TYPE,PRIOR WEIGHT,PRIOR
ENDPERIOD,DATE,ENDPERIOD,,TIME,DIMENSION,,,,True,,False,False,,,,,,
CHANNEL,DIMENSION,CHANNEL,,MEDIA,DIMENSION,,,,False,,False,False,,,,,,
RETAILER,DIMENSION,RETAILER,,MEDIA,DIMENSION,,,,False,,False,False,,,,,,
OUTLET,DIMENSION,OUTLET,,MEDIA,DIMENSION,,,,False,,False,False,,,,,,
DEPENDENT_VARIABLE,RESPONSE,DEPENDENT VARIABLE,,SALES,METRIC,,,,False,,False,False,,,,inf,,
OLV_YT23_MEDIA,MEDIA,OLV_YT23_MEDIA,,MEDIA,DIG,OLV/YT,OLV/YT,OLV/YT23,True,INCREMENTAL,False,False,POSITIVE,0,inf,inf,,
FB_INST_MEDIA,MEDIA,FB_INST_MEDIA,,MEDIA,DIG,FB/INST,FB/INST,FB/INST_MEDIA,True,INCREMENTAL,False,False,POSITIVE,0,inf,inf,,
ERETAIL_AMZ_MEDIA,MEDIA,ERETAIL_AMZ_MEDIA,,MEDIA,DIG,ERETAIL_AMAZ,ERETAIL_AMAZ,ERETAIL_AMZ_MEDIA,True,INCREMENTAL,False,False,POSITIVE,0,inf,inf,,
SRCH_GOO_METRIC,MEDIA,SRCH_GOO_METRIC,,MEDIA,DIG,SEARCH,GOOGLE,SEARCH_GOOGLE,True,INCREMENTAL,False,False,POSITIVE,0,inf,inf,,
SOC_INTL_MEDIA,MEDIA,SOC_INTL_MEDIA,,MEDIA,DIG,SOCIAL,INTL,SOCIAL_INTL_MEDIA,True,INCREMENTAL,False,False,POSITIVE,0,inf,inf,,
"""

legacy_payload = {
    "algorithm": {"type": "least_squares"},
    "errorallocation": {"activate": True},
    "spec": {
        "time_dim": "ENDPERIOD",
        "dep_var": "DEPENDENT_VARIABLE",
        "prior": True,
        "prior_weight": 1_000_000,
        "output": "NONE",
    },
    "params": {
        "GLMVSgibbsAlgorithm": {
            "NonInformativePriorsLogic": 6,
            "seedingLogic": "AllZeros",
            "MinIterations": 2000,
            "MaxIterations": 6000,
            "MinConvergenceIterations": 1000,
            "ConvergenceThreshold": 0.001,
            "CoefficientStatistics": True,
        },
        "LeastSquaresAlgorithm": {
            "CoefficientStatistics": True,
        },
        "InteriorPoint": {
            "SLESolverType": "pardiso",
        },
    },
    "variable_config": variable_config_csv,
}

LEGACY_MODEL_ID = "legacy_ls_raw"
LEGACY_ENV = "legacy"

legacy_registration = client.post(
    "/registry/configs",
    json={"env": LEGACY_ENV, "model_id": LEGACY_MODEL_ID, "config": legacy_payload},
).json()
legacy_registration


{'model_id': 'legacy_ls_raw',
 'environment': 'legacy',
 'spec': {'model_id': 'legacy_ls_raw',
  'name': 'Least Squares Model',
  'version': '0.0.1',
  'kind': 'pipeline',
  'hyperparams_schema': {'type': 'object',
   'properties': {'GLMVSgibbsAlgorithm.NonInformativePriorsLogic': {'title': 'GLMVSgibbsAlgorithm.NonInformativePriorsLogic',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 6},
    'GLMVSgibbsAlgorithm.seedingLogic': {'title': 'GLMVSgibbsAlgorithm.seedingLogic',
     'description': 'Imported from legacy params',
     'type': 'string',
     'default': 'AllZeros'},
    'GLMVSgibbsAlgorithm.MinIterations': {'title': 'GLMVSgibbsAlgorithm.MinIterations',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 2000},
    'GLMVSgibbsAlgorithm.MaxIterations': {'title': 'GLMVSgibbsAlgorithm.MaxIterations',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 6000},
    'GLMVSgibbs

## Inspect the Converted Configuration

In [113]:
import json

converted_configs = client.get(
    f"/registry/models/{LEGACY_MODEL_ID}/configs",
    params={"env": LEGACY_ENV},
).json()
if not converted_configs:
    raise RuntimeError("No configurations returned for legacy payload")
entry = converted_configs[0]
converted_config = entry.get("config")
if not converted_config:
    raise RuntimeError("Config response payload missing 'config' key")
print(f"Environment: {entry.get('environment')}")
print("Model section:")
print(json.dumps(converted_config.get("model", {}), indent=2))
variables = converted_config.get("variables", {}).get("definitions", [])
print("Variables (names -> include flag):")
print({row["name"]: row.get("include", True) for row in variables})


Environment: legacy
Model section:
{
  "id": "legacy_ls_raw",
  "kind": "pipeline",
  "name": "Least Squares Model",
  "tags": [
    "legacy-import",
    "uses-priors"
  ],
  "version": "0.0.1",
  "metadata": {
    "legacy_spec": {
      "prior": true,
      "output": "NONE",
      "dep_var": "DEPENDENT_VARIABLE",
      "time_dim": "ENDPERIOD",
      "prior_weight": 1000000
    },
    "legacy_algorithm": {
      "type": "least_squares"
    },
    "legacy_error_allocation": {
      "activate": true
    }
  }
}
Variables (names -> include flag):
{'ENDPERIOD': True, 'CHANNEL': False, 'RETAILER': False, 'OUTLET': False, 'DEPENDENT_VARIABLE': False, 'OLV_YT23_MEDIA': True, 'FB_INST_MEDIA': True, 'ERETAIL_AMZ_MEDIA': True, 'SRCH_GOO_METRIC': True, 'SOC_INTL_MEDIA': True}


## Augment the Config with a Library Binding

In [114]:
import copy

BINDING_MODEL_ID = "legacy_ls_binding"
BINDING_ENV = "demo"

if not converted_config:
    raise RuntimeError("converted_config is empty; register config first")

binding_config = copy.deepcopy(converted_config)
model_section = binding_config.setdefault("model", {})
model_section["id"] = BINDING_MODEL_ID
model_section["name"] = "Legacy Least Squares (Binding)"
model_section["kind"] = "library"
model_section["binding"] = {
    "kind": "library",
    "module": "model_zoo.examples.models",
    "callable": "linreg_train_and_infer",
}
model_section["inputs"] = ["X", "y"]

hyperparameters = binding_config.setdefault("hyperparameters", {})
schema_rows = hyperparameters.setdefault("schema", [])
if not any(row.get("name") == "fit_intercept" for row in schema_rows):
    schema_rows.append({"name": "fit_intercept", "dtype": "bool", "default": True, "description": "Whether to fit an intercept"})

value_rows = hyperparameters.setdefault("values", [])
updated = False
for row in value_rows:
    row["model_id"] = BINDING_MODEL_ID
    row["env"] = BINDING_ENV
    if row.get("param") == "fit_intercept":
        updated = True
if not updated:
    value_rows.append({"model_id": BINDING_MODEL_ID, "param": "fit_intercept", "value": True, "env": BINDING_ENV})

variables_section = binding_config.setdefault("variables", {})
definitions = variables_section.setdefault("definitions", [])
for definition in definitions:
    source = definition.get("source")
    if source == "DEPENDENT_VARIABLE":
        definition["name"] = "sales"
        definition.setdefault("metadata", {})["original_name"] = "DEPENDENT_VARIABLE"
        definition["include"] = True
    elif source in {"ENDPERIOD", "OLV_YT23_MEDIA"}:
        definition["include"] = True
    else:
        definition["include"] = False

existing_names = {definition.get("name") for definition in definitions}
if "X" not in existing_names:
    definitions.append({
        "name": "X",
        "source": "OLV_YT23_MEDIA",
        "include": True,
        "metadata": {"role": "feature"},
    })
if "y" not in existing_names:
    definitions.append({
        "name": "y",
        "source": "DEPENDENT_VARIABLE",
        "include": True,
        "metadata": {"role": "target"},
    })

binding_config["metrics"] = [
    {
        "metric_id": "r2_v1",
        "name": "R2_Score",
        "version": "1.0.0",
        "direction": "higher_is_better",
        "kind": "library",
        "binding": {"module": "model_zoo.examples.metrics", "callable": "r2_score"},
    },
    {
        "metric_id": "mse_v1",
        "name": "MeanSquaredError",
        "version": "1.0.0",
        "direction": "lower_is_better",
        "kind": "library",
        "binding": {"module": "model_zoo.examples.metrics", "callable": "mse"},
    },
]

binding_config["evaluation"] = {
    "metric_ids": ["r2_v1", "mse_v1"],
    "y_true_key": "y",
    "compute_train": True,
    "compute_valid": True,
}

binding_config["context"] = {"timeout_s": 30}

print("Prepared binding-enabled config:")
print(json.dumps({"model": model_section, "variables": variables_section}, indent=2))


Prepared binding-enabled config:
{
  "model": {
    "id": "legacy_ls_binding",
    "kind": "library",
    "name": "Legacy Least Squares (Binding)",
    "tags": [
      "legacy-import",
      "uses-priors"
    ],
    "version": "0.0.1",
    "metadata": {
      "legacy_spec": {
        "prior": true,
        "output": "NONE",
        "dep_var": "DEPENDENT_VARIABLE",
        "time_dim": "ENDPERIOD",
        "prior_weight": 1000000
      },
      "legacy_algorithm": {
        "type": "least_squares"
      },
      "legacy_error_allocation": {
        "activate": true
      }
    },
    "binding": {
      "kind": "library",
      "module": "model_zoo.examples.models",
      "callable": "linreg_train_and_infer"
    },
    "inputs": [
      "X",
      "y"
    ]
  },
  "variables": {
    "groups": [],
    "definitions": [
      {
        "name": "ENDPERIOD",
        "source": "ENDPERIOD",
        "include": true,
        "metadata": {
          "type": "DIMENSION",
          "include": true,
 

## Register the Binding-Ready Config

In [115]:
binding_registration = client.post(
    "/registry/configs",
    json={"env": BINDING_ENV, "model_id": BINDING_MODEL_ID, "config": binding_config},
).json()
binding_registration


{'model_id': 'legacy_ls_binding',
 'environment': 'demo',
 'spec': {'model_id': 'legacy_ls_binding',
  'name': 'Legacy Least Squares (Binding)',
  'version': '0.0.1',
  'kind': 'library',
  'hyperparams_schema': {'type': 'object',
   'properties': {'GLMVSgibbsAlgorithm.NonInformativePriorsLogic': {'title': 'GLMVSgibbsAlgorithm.NonInformativePriorsLogic',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 6},
    'GLMVSgibbsAlgorithm.seedingLogic': {'title': 'GLMVSgibbsAlgorithm.seedingLogic',
     'description': 'Imported from legacy params',
     'type': 'string',
     'default': 'AllZeros'},
    'GLMVSgibbsAlgorithm.MinIterations': {'title': 'GLMVSgibbsAlgorithm.MinIterations',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 2000},
    'GLMVSgibbsAlgorithm.MaxIterations': {'title': 'GLMVSgibbsAlgorithm.MaxIterations',
     'description': 'Imported from legacy params',
     'type': 'int',
     'default': 6000},

## Author a Canonical Configuration

In [116]:
CANONICAL_MODEL_ID = "legacy_ls_manual"
CANONICAL_ENV = "manual"
canonical_config = {
    "model": {
        "id": CANONICAL_MODEL_ID,
        "name": "Legacy Least Squares (Manual Canonical)",
        "version": "1.0.0",
        "kind": "library",
        "binding": {
            "kind": "library",
            "module": "model_zoo.examples.models",
            "callable": "linreg_train_and_infer",
        },
        "inputs": ["X", "y"],
    },
    "hyperparameters": {
        "schema": [
            {"name": "fit_intercept", "dtype": "bool", "default": True, "description": "Whether to fit an intercept"}
        ],
        "values": [
            {"model_id": CANONICAL_MODEL_ID, "param": "fit_intercept", "value": True, "env": CANONICAL_ENV}
        ],
    },
    "variables": {
        "definitions": [
            {"name": "week_end", "source": "ENDPERIOD", "include": False},
            {"name": "digital_spend", "source": "OLV_YT23_MEDIA", "include": False, "metadata": {"channel": "OLV/YT"}},
            {"name": "sales", "source": "DEPENDENT_VARIABLE", "include": False},
            {"name": "X", "source": "OLV_YT23_MEDIA", "include": True, "metadata": {"role": "feature"}},
            {"name": "y", "source": "DEPENDENT_VARIABLE", "include": True, "metadata": {"role": "target"}},
        ],
        "groups": [],
    },
    "metrics": [
        {
            "metric_id": "r2_v1",
            "name": "R2_Score",
            "version": "1.0.0",
            "direction": "higher_is_better",
            "kind": "library",
            "binding": {"module": "model_zoo.examples.metrics", "callable": "r2_score"},
            "metadata": {"env": CANONICAL_ENV},
        },
        {
            "metric_id": "mse_v1",
            "name": "MeanSquaredError",
            "version": "1.0.0",
            "direction": "lower_is_better",
            "kind": "library",
            "binding": {"module": "model_zoo.examples.metrics", "callable": "mse"},
            "metadata": {"env": CANONICAL_ENV},
        },
    ],
    "evaluation": {
        "metric_ids": ["r2_v1", "mse_v1"],
        "y_true_key": "y",
        "compute_train": True,
        "compute_valid": True,
    },
    "context": {"timeout_s": 30},
}
print("Canonical configuration staged for env:", CANONICAL_ENV)


Canonical configuration staged for env: manual


## Register the Canonical Config

In [117]:
canonical_registration = client.post(
    "/registry/configs",
    json={"env": CANONICAL_ENV, "model_id": CANONICAL_MODEL_ID, "config": canonical_config},
).json()
canonical_registration


{'model_id': 'legacy_ls_manual',
 'environment': 'manual',
 'spec': {'model_id': 'legacy_ls_manual',
  'name': 'Legacy Least Squares (Manual Canonical)',
  'version': '1.0.0',
  'kind': 'library',
  'hyperparams_schema': {'type': 'object',
   'properties': {'fit_intercept': {'title': 'fit_intercept',
     'description': 'Whether to fit an intercept',
     'type': 'boolean',
     'default': True}},
   'required': [],
   'additionalProperties': True},
  'tags': [],
  'metadata': {},
  'dependencies': []},
 'stored': True}

## Derive an Experiment Configuration

In [118]:
import copy

EXPERIMENT_MODEL_ID = "legacy_ls_experiment"
EXPERIMENT_ENV = "experiment"
experiment_config = copy.deepcopy(canonical_config)
experiment_config["model"]["id"] = EXPERIMENT_MODEL_ID
experiment_config["model"]["name"] = "Legacy Least Squares (Experiment)"
experiment_config["hyperparameters"]["values"] = [
    {"model_id": EXPERIMENT_MODEL_ID, "param": "fit_intercept", "value": False, "env": EXPERIMENT_ENV}
]
experiment_config["metrics"] = [
    {
        "metric_id": "mse_v1",
        "name": "MeanSquaredError",
        "version": "1.0.0",
        "direction": "lower_is_better",
        "kind": "library",
        "binding": {"module": "model_zoo.examples.metrics", "callable": "mse"},
        "metadata": {"env": EXPERIMENT_ENV},
    }
]
experiment_config["evaluation"] = {
    "metric_ids": ["mse_v1"],
    "y_true_key": "y",
    "compute_train": True,
    "compute_valid": True,
}
print("Experiment configuration staged for env:", EXPERIMENT_ENV)


Experiment configuration staged for env: experiment


## Register the Experiment Config

In [119]:
experiment_registration = client.post(
    "/registry/configs",
    json={"env": EXPERIMENT_ENV, "model_id": EXPERIMENT_MODEL_ID, "config": experiment_config},
).json()
experiment_registration


{'model_id': 'legacy_ls_experiment',
 'environment': 'experiment',
 'spec': {'model_id': 'legacy_ls_experiment',
  'name': 'Legacy Least Squares (Experiment)',
  'version': '1.0.0',
  'kind': 'library',
  'hyperparams_schema': {'type': 'object',
   'properties': {'fit_intercept': {'title': 'fit_intercept',
     'description': 'Whether to fit an intercept',
     'type': 'boolean',
     'default': True}},
   'required': [],
   'additionalProperties': True},
  'tags': [],
  'metadata': {},
  'dependencies': []},
 'stored': True}

## Submit Runs

In [120]:
import time
import json
from requests import HTTPError

data_sources = {
    "ENDPERIOD": ["2024-01-07", "2024-01-14", "2024-01-21", "2024-01-28", "2024-02-04", "2024-02-11"],
    "OLV_YT23_MEDIA": [100.0, 120.0, 110.0, 130.0, 125.0, 140.0],
    "DEPENDENT_VARIABLE": [210.0, 238.0, 225.0, 250.0, 245.0, 265.0],
}

run_scenarios = [
    {"label": "binding", "model_id": BINDING_MODEL_ID, "env": BINDING_ENV, "params": {"fit_intercept": True}},
    {"label": "manual", "model_id": CANONICAL_MODEL_ID, "env": CANONICAL_ENV, "params": {"fit_intercept": True}},
    {"label": "experiment", "model_id": EXPERIMENT_MODEL_ID, "env": EXPERIMENT_ENV, "params": {"fit_intercept": False}},
]

run_results: Dict[str, Dict[str, Any]] = {}
run_ids: Dict[str, str] = {}
TERMINAL_STATES = {"succeeded", "failed", "cancelled"}

for scenario in run_scenarios:
    payload = {
        "model_id": scenario["model_id"],
        "env": scenario["env"],
        "data_sources": data_sources,
        "params": scenario.get("params", {}),
    }
    try:
        submission = client.post("/runs", json=payload).json()
    except HTTPError as exc:
        detail = getattr(exc.response, "text", str(exc))
        run_results[scenario["label"]] = {
            "status": "submission_failed",
            "error": detail,
            "run_id": None,
        }
        print(
            f"Run submission failed for {scenario['label']} "
            f"(model={scenario['model_id']}, env={scenario['env']}): {detail}"
        )
        continue

    run_id = submission.get("run_id")
    if not run_id:
        run_results[scenario["label"]] = {
            "status": "submission_failed",
            "error": submission,
            "run_id": None,
        }
        print(
            f"Unexpected submission payload for {scenario['label']}: "
            f"{json.dumps(submission, indent=2)}"
        )
        continue

    run_ids[scenario["label"]] = run_id
    status_resp = client.get(f"/runs/{run_id}").json()
    while str(status_resp.get("status", "")).lower() not in TERMINAL_STATES:
        time.sleep(0.3)
        status_resp = client.get(f"/runs/{run_id}").json()

    status_lower = str(status_resp.get("status", "")).lower()
    if "result" in status_resp:
        payload_to_store = status_resp
    else:
        payload_to_store = {"status": status_resp.get("status"), "error": status_resp.get("error")}
    payload_to_store["run_id"] = run_id
    run_results[scenario["label"]] = payload_to_store

    print(
        f"Completed run for {scenario['label']} (model={scenario['model_id']}, env={scenario['env']}) "
        f"with status {status_resp.get('status')}"
    )


Completed run for binding (model=legacy_ls_binding, env=demo) with status succeeded
Completed run for manual (model=legacy_ls_manual, env=manual) with status succeeded
Completed run for experiment (model=legacy_ls_experiment, env=experiment) with status succeeded


## Run Results

In [121]:
import json

for label, payload in run_results.items():
    status = payload.get('status')
    run_id = payload.get('run_id') or run_ids.get(label)
    print(f"=== Scenario: {label} ===")
    print(f"Status : {status}")
    if run_id:
        print(f"Run ID: {run_id}")

    error_detail = payload.get('error')
    if status != 'succeeded':
        if error_detail:
            print("Error detail:")
            if isinstance(error_detail, (dict, list)):
                print(json.dumps(error_detail, indent=2))
            else:
                print(error_detail)
        else:
            print("Full payload:")
            print(json.dumps(payload, indent=2))
        print()
        continue

    result = payload.get('result') or {}
    predictions = result.get('predictions')
    if isinstance(predictions, list):
        preview = predictions[:5]
        total = len(predictions)
        print(f"Predictions preview (first {len(preview)} of {total}): {preview}")
    elif predictions is not None:
        print(f"Predictions: {predictions}")

    if result.get('eval_train'):
        print("Train metrics:")
        print(json.dumps(result['eval_train'], indent=2))
    if result.get('eval_valid'):
        print("Validation metrics:")
        print(json.dumps(result['eval_valid'], indent=2))

    diagnostics = result.get('diagnostics')
    if diagnostics:
        print("Diagnostics:")
        print(json.dumps(diagnostics, indent=2))

    summary = result.get('summary_text') or result.get('summary')
    if summary:
        print("Summary:")
        if isinstance(summary, str):
            print(summary)
        else:
            print(json.dumps(summary, indent=2))
    print()


=== Scenario: binding ===
Status : succeeded
Run ID: 9adbe808-3af0-4f35-8dc9-9f3fa8e15fdc
Predictions preview (first 5 of 6): [210.65306122449, 237.7061224489796, 224.1795918367348, 251.23265306122445, 244.46938775510205]
Train metrics:
{
  "r2_v1": 0.9969491332375254,
  "mse_v1": 0.6763390254060746
}
Validation metrics:
{
  "r2_v1": 0.9983022907122029,
  "mse_v1": 0.16977092877970754
}
Diagnostics:
{
  "coef[x]": 1.3526530612244818,
  "intercept": 75.38775510204181
}
Summary:
Call:
  model_zoo.examples.models::linreg_train_and_infer(
    GLMVSgibbsAlgorithm.NonInformativePriorsLogic = 6
    GLMVSgibbsAlgorithm.seedingLogic = 'AllZeros'
    GLMVSgibbsAlgorithm.MinIterations = 2000
    GLMVSgibbsAlgorithm.MaxIterations = 6000
    GLMVSgibbsAlgorithm.MinConvergenceIterations = 1000
    GLMVSgibbsAlgorithm.ConvergenceThreshold = 0.001
    GLMVSgibbsAlgorithm.CoefficientStatistics = True
    LeastSquaresAlgorithm.CoefficientStatistics = True
    InteriorPoint.SLESolverType = 'pardiso'
    